# 02. Document Processing
— DoclingLoader `DOC_CHUNKS` 기반 Structure-aware Chunking

## 실습 목적

`01_naive_rag_limits_docling_markdown.ipynb`에서는 다음과 같이 처리했다.

```text
PDF
 ↓
DoclingLoader
 ↓
ExportType.MARKDOWN
 ↓
RecursiveCharacterTextSplitter
 ↓
Naive Chunk
 ↓
Dense Retrieval
```

이번 실습에서는 Docling의 장점을 더 적극적으로 활용한다.

```text
PDF
 ↓
DoclingLoader
 ↓
ExportType.DOC_CHUNKS
 ↓
HybridChunker
 ↓
문서 구조 + Token 기반 Chunk
 ↓
Metadata
 ↓
Dense Retrieval
```

즉, **문서를 Markdown으로 만든 뒤 다시 문자열 기준으로 자르는 것이 아니라**,  
Docling이 인식한 제목, 문단, 표, caption 등의 구조를 유지하면서 `HybridChunker`가 직접 Chunk를 생성하도록 한다.

## 학습 목표

실습 후 다음 내용을 설명할 수 있다.

1. `ExportType.MARKDOWN`과 `ExportType.DOC_CHUNKS`의 차이
2. `HybridChunker`가 문서 구조와 token 크기를 함께 고려하는 이유
3. `DoclingLoader`가 Chunking까지 수행하도록 구성하는 방법
4. heading, caption, page provenance가 RAG에서 중요한 이유
5. Docling metadata를 Vector DB용 단순 metadata로 정규화하는 방법
6. 1번 Naive Chunking과 2번 Structure-aware Chunking의 차이

## 실습 데이터

`data/AI@Data_Report_2026_전망_251223(최종).pdf`


## 1. 환경 설정

필요 패키지 예시는 다음과 같다.

```bash
uv add langchain-docling langchain-chroma langchain-openai tiktoken python-dotenv
uv add "docling-core[chunking-openai]"
```

이번 실습에서는 OpenAI `text-embedding-3-small`과 같은 tokenizer 체계를 사용하기 위해  
`tiktoken` 기반 `OpenAITokenizer`를 `HybridChunker`에 전달한다.

> 핵심은 **Chunking 기준 tokenizer와 Embedding 단계의 tokenizer 계열을 가능한 한 맞추는 것**이다.


In [1]:
import sys
from pathlib import Path

# 상위 폴더의 common_config.py를 import 할 수 있도록 경로 추가
sys.path.append("..")

# 공통 설정에서 임베딩 모델을 불러온다.
from common_config import embedding_model


# ---------------------------------------------------------
# 실습 대상 PDF
# ---------------------------------------------------------
PDF_PATH = Path("data/AI@Data_Report_2026_전망_251223(최종).pdf")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {PDF_PATH}")

print(f"PDF: {PDF_PATH}")


PDF: data/AI@Data_Report_2026_전망_251223(최종).pdf


# 2. PDF 파싱 옵션 구성

1번과 2번의 비교에서 중요한 점은 **PDF 파싱 조건은 동일하게 유지하고 Chunking 방식만 바꾸는 것**이다.

이번 실습에서는 다음 설정을 사용한다.

- OCR 비활성화: 텍스트 기반 PDF이므로 불필요한 OCR을 수행하지 않음
- Table Structure 활성화: 표의 행·열·셀 구조를 인식
- Heading Hierarchy 활성화: PDF의 절/소절 계층을 가능한 한 복원
- Parsed Page 유지: heading hierarchy의 글꼴 스타일 기반 추론에 활용

이렇게 하면 이후 `HybridChunker`가 문서의 heading 구조를 더 효과적으로 활용할 수 있다.


In [2]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    HeadingHierarchyOptions,
    PdfPipelineOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption


# ---------------------------------------------------------
# 1. PDF Pipeline 설정
# ---------------------------------------------------------
pipeline_options = PdfPipelineOptions()

# 현재 보고서는 텍스트 기반 PDF이므로 OCR을 사용하지 않는다.
# 스캔 문서라면 True로 변경할 수 있다.
pipeline_options.do_ocr = False

# 표의 행/열/셀 구조를 인식한다.
pipeline_options.do_table_structure = True

# PDF 제목의 계층(Level)을 추론하도록 활성화한다.
pipeline_options.heading_hierarchy_options = HeadingHierarchyOptions(
    enabled=True
)

# 글꼴 크기/스타일 정보까지 heading hierarchy 추론에 사용하려면
# parsed page 정보를 유지해야 한다.
pipeline_options.generate_parsed_pages = True


# ---------------------------------------------------------
# 2. Docling DocumentConverter 구성
# ---------------------------------------------------------
# DoclingLoader가 내부에서 이 converter를 사용하도록 전달한다.
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

print("DocumentConverter 구성 완료")


DocumentConverter 구성 완료


# 3. HybridChunker 구성

`HybridChunker`는 크게 두 단계를 결합한다.

### 1) 문서 구조 기반 Chunking

먼저 Docling이 인식한 다음 구조를 기준으로 Chunk 후보를 만든다.

- Heading
- Paragraph
- Table
- Caption
- 동일한 Section의 문서 요소

### 2) Token-aware 보정

구조 단위가 너무 크면 token 한도에 맞게 추가 분할하고,  
같은 heading 아래의 작은 인접 chunk는 가능한 경우 병합한다.

즉 다음 차이가 있다.

```text
RecursiveCharacterTextSplitter
→ 주로 문자열 길이를 기준으로 분할

HybridChunker
→ 문서 구조를 먼저 고려
→ token 크기로 추가 보정
```


In [3]:
import tiktoken

from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer


# ---------------------------------------------------------
# Embedding 모델과 Chunking tokenizer 설정
# ---------------------------------------------------------
EMBED_MODEL_NAME = "text-embedding-3-small"

# 실습에서는 검색 단위가 지나치게 커지지 않도록
# 하나의 chunk를 최대 700 token 정도로 제한한다.
MAX_TOKENS = 700


# OpenAI embedding 모델에서 사용하는 tiktoken encoding을 불러온다.
encoding = tiktoken.encoding_for_model(EMBED_MODEL_NAME)


# Docling HybridChunker가 token 수를 계산할 수 있도록
# tiktoken encoding을 Docling용 tokenizer 객체로 감싼다.
tokenizer = OpenAITokenizer(
    tokenizer=encoding,
    max_tokens=MAX_TOKENS,
)


# ---------------------------------------------------------
# HybridChunker
# ---------------------------------------------------------
chunker = HybridChunker(
    tokenizer=tokenizer,

    # 같은 heading 아래의 작은 인접 chunk를
    # token 한도 내에서 합칠 수 있도록 한다.
    merge_peers=True,

    # 표가 token 한도로 인해 여러 chunk로 나뉠 경우
    # 각 chunk에서 표 header의 의미가 유지되도록 한다.
    repeat_table_header=True,
)

print("HybridChunker 구성 완료")
print(f"Embedding tokenizer : {EMBED_MODEL_NAME}")
print(f"Chunk 최대 token   : {MAX_TOKENS}")


HybridChunker 구성 완료
Embedding tokenizer : text-embedding-3-small
Chunk 최대 token   : 700


# 4. DoclingLoader + DOC_CHUNKS

이제 `DoclingLoader`에 다음 세 가지를 전달한다.

```text
converter
+
ExportType.DOC_CHUNKS
+
HybridChunker
```

`DOC_CHUNKS`를 사용하면 `DoclingLoader`가 내부적으로 다음 작업을 수행한다.

```text
PDF
 ↓
DocumentConverter
 ↓
DoclingDocument
 ↓
HybridChunker.chunk()
 ↓
HybridChunker.contextualize()
 ↓
LangChain Document[]
```

따라서 이번 실습에서는 다음 작업을 **직접 다시 수행하지 않는다.**

```python
chunker.chunk(...)
chunker.contextualize(...)
RecursiveCharacterTextSplitter(...)
```

`loader.load()`의 결과가 바로 여러 개의 LangChain `Document`이다.


In [4]:
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType


# ---------------------------------------------------------
# DoclingLoader 구성
# ---------------------------------------------------------
loader = DoclingLoader(
    file_path=str(PDF_PATH),

    # 앞에서 설정한 PDF 파싱 옵션을 그대로 사용
    converter=converter,

    # 문서 전체 Markdown이 아니라
    # Docling Chunk 단위의 LangChain Document를 반환
    export_type=ExportType.DOC_CHUNKS,

    # Chunking 전략을 명시적으로 지정
    chunker=chunker,
)


# ---------------------------------------------------------
# PDF → Chunked LangChain Documents
# ---------------------------------------------------------
raw_chunks = loader.load()

print(f"생성된 LangChain Document 수: {len(raw_chunks)}")


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


생성된 LangChain Document 수: 48


## 4.1 DoclingLoader가 반환한 Chunk 확인

`DOC_CHUNKS` 모드에서는 각 `Document`의 `page_content`가  
`HybridChunker.contextualize()`를 거친 검색용 텍스트이다.

또한 `metadata["dl_meta"]`에는 Docling의 구조 정보가 포함된다.

대표적으로 다음 정보를 확인할 수 있다.

- `headings`: 해당 Chunk가 속한 heading 경로
- `captions`: 표/그림 caption
- `doc_items`: 원본 문서 요소
- `prov`: page 번호와 bounding box 등의 provenance
- `origin`: 원본 문서 정보

먼저 Docling이 반환한 원본 metadata를 직접 확인한다.


In [5]:
# 처음 몇 개 Chunk의 내용과 원본 metadata 확인
for i, doc in enumerate(raw_chunks[:3], start=1):
    print("=" * 100)
    print(f"[Chunk {i}]")
    print()
    print("[metadata]")
    print(doc.metadata)
    print()
    print("[page_content]")
    print(doc.page_content[:800])
    print()


[Chunk 1]

[metadata]
{'source': 'data/AI@Data_Report_2026_전망_251223(최종).pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/2', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 383.911935, 't': 533.3442246, 'r': 450.49696855999997, 'b': 520.3983726, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 8]}]}], 'headings': ['토픽 분석을 통한 AI 주요 트렌드 및 2026 전망'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 5246665392694994170, 'filename': 'AI@Data_Report_2026_전망_251223(최종).pdf'}}}

[page_content]
토픽 분석을 통한 AI 주요 트렌드 및 2026 전망
2025-제3호

[Chunk 2]

[metadata]
{'source': 'data/AI@Data_Report_2026_전망_251223(최종).pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/4', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_it

# 5. Docling Metadata 정규화

Docling의 `dl_meta`는 문서 구조와 provenance를 풍부하게 담고 있지만  
그대로 Vector DB metadata에 저장하기에는 중첩 구조가 복잡하다.

따라서 실습에서는 검색·필터링·출처 표시에서 자주 사용할 값만 평탄화한다.

저장할 metadata는 다음과 같다.

| 필드 | 의미 |
|---|---|
| `source` | 원본 PDF |
| `chunk_id` | Chunk 식별 번호 |
| `page` | 대표 페이지 |
| `pages` | Chunk가 걸쳐 있는 전체 페이지 |
| `heading` | Heading breadcrumb |
| `caption` | 표/그림 Caption |
| `num_tokens` | 실제 `page_content`의 token 수 |
| `parser` | 문서 파서 |
| `chunker` | Chunking 방식 |

> `raw_chunks`는 Docling 원본 metadata를 확인할 때 사용하고,  
> `chunks`는 ChromaDB 저장에 적합하도록 metadata를 단순화한 버전으로 사용한다.


In [6]:
from langchain_core.documents import Document


def extract_pages(dl_meta: dict) -> list[int]:
    """Docling dl_meta의 provenance에서 페이지 번호를 추출한다."""

    pages = set()

    # dl_meta["doc_items"]에는 Chunk를 구성하는 원본 문서 요소가 들어 있다.
    for item in dl_meta.get("doc_items", []):
        for prov in item.get("prov", []):
            page_no = prov.get("page_no")

            if page_no is not None:
                pages.add(page_no)

    return sorted(pages)


def normalize_docling_document(doc: Document, chunk_id: int) -> Document:
    """Docling metadata를 ChromaDB에서 다루기 쉬운 형태로 평탄화한다."""

    # DoclingLoader가 생성한 구조 metadata
    dl_meta = doc.metadata.get("dl_meta", {})

    pages = extract_pages(dl_meta)
    headings = dl_meta.get("headings") or []
    captions = dl_meta.get("captions") or []

    # page_content는 이미 HybridChunker.contextualize() 결과이므로
    # 다시 contextualize()를 호출할 필요가 없다.
    text = doc.page_content

    metadata = {
        "source": doc.metadata.get("source", str(PDF_PATH)),
        "chunk_id": chunk_id,

        # 첫 번째 페이지를 대표 페이지로 사용
        "page": pages[0] if pages else -1,

        # Chroma metadata에서는 list보다 문자열 형태가 다루기 편하다.
        "pages": ",".join(map(str, pages)),

        # Heading breadcrumb
        "heading": " > ".join(map(str, headings)),

        # 표/그림 Caption
        "caption": " | ".join(map(str, captions)),

        # 실제 Embedding 입력 문자열 기준 token 수
        "num_tokens": tokenizer.count_tokens(text),

        "parser": "docling",
        "chunker": "HybridChunker",
    }

    return Document(
        page_content=text,
        metadata=metadata,
    )


# ---------------------------------------------------------
# Vector DB 저장용 metadata로 정규화
# ---------------------------------------------------------
chunks = [
    normalize_docling_document(doc, chunk_id=i)
    for i, doc in enumerate(raw_chunks, start=1)
]

print(f"정규화된 LangChain Document 수: {len(chunks)}")


정규화된 LangChain Document 수: 48


## 5.1 정규화된 Metadata 확인

Chunk마다 heading과 page 정보가 어떻게 보존되어 있는지 확인한다.

이 정보는 이후 다음 단계에서 활용할 수 있다.

- 검색 결과 디버깅
- 출처 표시
- Citation 생성
- 특정 페이지/Section 필터링
- Retrieval 평가


In [7]:
for doc in chunks[:5]:
    print("=" * 100)
    print(doc.metadata)
    print("-" * 100)
    print(doc.page_content[:600])
    print()


{'source': 'data/AI@Data_Report_2026_전망_251223(최종).pdf', 'chunk_id': 1, 'page': 1, 'pages': '1', 'heading': '토픽 분석을 통한 AI 주요 트렌드 및 2026 전망', 'caption': '', 'num_tokens': 37, 'parser': 'docling', 'chunker': 'HybridChunker'}
----------------------------------------------------------------------------------------------------
토픽 분석을 통한 AI 주요 트렌드 및 2026 전망
2025-제3호

{'source': 'data/AI@Data_Report_2026_전망_251223(최종).pdf', 'chunk_id': 2, 'page': 1, 'pages': '1', 'heading': '토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 > 목차', 'caption': '', 'num_tokens': 142, 'parser': 'docling', 'chunker': 'HybridChunker'}
----------------------------------------------------------------------------------------------------
토픽 분석을 통한 AI 주요 트렌드 및 2026 전망
목차
- ❶ 배경 및 목적 /p.06
- ❷ 자료 분석 및 방법 /p.09
- ❸ 분석 결과 /p.12
- ❹ 2026년 전망: AI 생태계 방향성 /p.17
- ❺ 분석 결과로 본 우리의 대응 및 노력 /p.20

{'source': 'data/AI@Data_Report_2026_전망_251223(최종).pdf', 'chunk_id': 3, 'page': 2, 'pages': '2', 'heading': 'AI@Data Report', 'caption': '', 'num_tokens': 

## 5.2 Chunk Token 크기 확인

`HybridChunker`는 모든 Chunk를 동일한 길이로 만드는 것이 목적이 아니다.

문서 구조를 우선 보존하면서 필요한 경우 token 한도에 맞게 조정하므로  
Chunk마다 token 수가 서로 다른 것이 자연스럽다.

다만 지나치게 작은 Chunk나 예상보다 큰 Chunk가 없는지는 확인하는 것이 좋다.


In [8]:
from statistics import mean, median


token_counts = [
    doc.metadata["num_tokens"]
    for doc in chunks
]

print(f"Chunk 수       : {len(token_counts)}")
print(f"최소 token     : {min(token_counts)}")
print(f"최대 token     : {max(token_counts)}")
print(f"평균 token     : {mean(token_counts):.1f}")
print(f"중앙값 token   : {median(token_counts):.1f}")


Chunk 수       : 48
최소 token     : 20
최대 token     : 696
평균 token     : 370.6
중앙값 token   : 370.5


# 6. ChromaDB 구성

1번의 Naive Chunk와 섞이지 않도록 별도의 Vector DB를 사용한다.

```text
01
MARKDOWN + RecursiveCharacterTextSplitter
→ chroma_db/ai_trend_report_naive_markdown

02
DOC_CHUNKS + HybridChunker
→ chroma_db/ai_trend_report_doc_chunks
```

검색 score를 이해하기 쉽도록 cosine distance를 명시적으로 사용한다.

- distance가 작을수록 질문과 가까움
- distance가 클수록 질문과 멀어짐


In [9]:
import chromadb

from langchain_chroma import Chroma


# ---------------------------------------------------------
# DOC_CHUNKS 전용 Vector DB
# ---------------------------------------------------------
DB_PATH = Path("chroma_db/ai_trend_report_doc_chunks")
COLLECTION_NAME = "ai_trend_report_2026_doc_chunks"


# Chunking 설정을 수정한 뒤 다시 실습할 때는 True로 둔다.
# 이미 생성된 DB를 유지하려면 False로 변경할 수 있다.
RESET_DB = True


# 폴더를 shutil.rmtree로 지우면 같은 커널에 캐시된 Chroma client가
# 삭제된 sqlite 파일을 계속 잡고 있어 "readonly database" 오류가 난다.
# 따라서 폴더는 그대로 두고 client를 통해 collection만 삭제한다.
client = chromadb.PersistentClient(path=str(DB_PATH))

if RESET_DB and COLLECTION_NAME in [col.name for col in client.list_collections()]:
    client.delete_collection(COLLECTION_NAME)
    print(f"기존 collection 삭제: {COLLECTION_NAME}")


# 공통 설정에서 Embedding 모델 초기화
embeddings = embedding_model()


# ---------------------------------------------------------
# Structure-aware Chunk를 ChromaDB에 저장
# ---------------------------------------------------------
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,

    # 위에서 만든 client를 그대로 사용
    client=client,

    # cosine distance 사용
    collection_metadata={
        "hnsw:space": "cosine"
    },
)


print(f"저장된 Chunk 수: {vector_store._collection.count()}")

기존 collection 삭제: ai_trend_report_2026_doc_chunks
저장된 Chunk 수: 48


# 7. Structure-aware Retrieval 확인

검색 결과를 단순히 텍스트만 보는 것이 아니라 다음을 함께 확인한다.

- cosine distance
- page
- heading
- caption
- chunk_id

이 과정을 통해 **검색 결과가 원본 문서의 어느 위치에서 왔는지** 추적할 수 있다.


In [10]:
def inspect_retrieval(query: str, k: int = 5):
    """질문에 대한 검색 결과와 구조 metadata를 함께 출력한다."""

    results = vector_store.similarity_search_with_score(
        query,
        k=k,
    )

    print("=" * 100)
    print(f"[질문] {query}")
    print(f"[검색 개수] k={k}")
    print("cosine distance: 낮을수록 더 유사함")
    print("=" * 100)

    for rank, (doc, score) in enumerate(results, start=1):
        meta = doc.metadata

        print(
            f"\n[{rank}] distance={score:.4f} "
            f"| chunk_id={meta.get('chunk_id')} "
            f"| pages={meta.get('pages')}"
        )

        print(f"heading : {meta.get('heading')}")

        if meta.get("caption"):
            print(f"caption : {meta.get('caption')}")

        print("-" * 100)
        print(doc.page_content[:700].replace("\n", " "))


## 7.1 제목/Section 구조가 중요한 질문

Heading 정보가 Chunk의 검색 문맥에 포함되므로  
해당 내용이 어느 주제/절에 속하는지 함께 반영될 수 있다.


In [11]:
query = "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?"

inspect_retrieval(
    query,
    k=5,
)


[질문] 2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
[검색 개수] k=5
cosine distance: 낮을수록 더 유사함

[1] distance=0.3376 | chunk_id=10 | pages=5
heading : AI@Data Report > ■ 2026년 AI 생태계 방향성 전망
----------------------------------------------------------------------------------------------------
AI@Data Report ■ 2026년 AI 생태계 방향성 전망 - 산업·기술·정책이 서로 다른 변화 축을 보이지만, 2026년에는 세 변화가 동시에 심화되며 AI 생태계의 구조 재편이 가속화될 가능성이 있음 - Ÿ 산업은 도입 확대, 기술은 기능 고도화, 정책은 책임·안전 중심의 제도화가 각각 진전될 것으로 예상되며, 이러한 흐름은 AI 활용의 범위, 속도, 규율 체계에 모두 영향을 미칠 것으로 전망됨

[2] distance=0.4124 | chunk_id=2 | pages=1
heading : 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 > 목차
----------------------------------------------------------------------------------------------------
토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 목차 - ❶ 배경 및 목적 /p.06 - ❷ 자료 분석 및 방법 /p.09 - ❸ 분석 결과 /p.12 - ❹ 2026년 전망: AI 생태계 방향성 /p.17 - ❺ 분석 결과로 본 우리의 대응 및 노력 /p.20

[3] distance=0.4193 | chunk_id=40 | pages=18
heading : 4.3 2026년 AI 기술 전망
-------------------------------------------------------------------------------------

## 7.2 표 구조가 중요한 질문

`repeat_table_header=True`는 하나의 큰 표가 token 한도로 인해 여러 Chunk로 나뉘는 경우  
각 Chunk가 column의 의미를 잃지 않도록 표 header를 반복하는 데 도움을 준다.


In [12]:
query = "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?"

inspect_retrieval(
    query,
    k=5,
)


[질문] 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
[검색 개수] k=5
cosine distance: 낮을수록 더 유사함

[1] distance=0.4163 | chunk_id=21 | pages=10
heading : 분석 절차 CHAPTER2 > 3 2.1 자료 수집 절차 및 분석 과정 > 결과분석
----------------------------------------------------------------------------------------------------
분석 절차 CHAPTER2 3 2.1 자료 수집 절차 및 분석 과정 결과분석 < 표 2 > 분야별 데이터 수집 키워드  AI산업, 사용 키워드 = AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환. AI기술, 사용 키워드 = AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발. AI정책, 사용 키워드 = AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책 [ 그림 1 ] 분야별 데이터셋 구성 예시

[2] distance=0.4840 | chunk_id=19 | pages=9
heading : 분석 절차 CHAPTER2 > 3 2.1 자료 수집 절차 및 분석 과정 > 결과분석
----------------------------------------------------------------------------------------------------
분석 절차 CHAPTER2 3 2.1 자료 수집 절차 및 분석 과정 결과분석 토픽·키워드 패턴을 분석해 2025년 AI 주요 이슈와 향후 변화 방향 도출 - 본 분석은 2025년 1월부터 11월까지 매주 산업·기술·정책 세 분야별로 국내 매체 1건, 해외 매체 1건씩 총 6건을 선정하여, 정기적인 기사 및 보고서 스크랩을 통해 분석에 필요한 데이터셋을 누적하여 확보함 - Ÿ 연구기관·글로벌 기술기업·전문지·경제지·정부기관·국제기구

## 7.3 Metadata를 활용한 출처 추적

RAG에서는 검색된 텍스트만 확보하는 것보다  
**그 텍스트가 어디에서 왔는지 추적할 수 있는 정보**를 함께 보존하는 것이 중요하다.

이번 실습에서는 page와 heading을 이용해 검색 결과의 출처를 확인한다.


In [13]:
query = "추론형 데이터는 어떤 유형으로 구분되나요?"

results = vector_store.similarity_search(
    query,
    k=5,
)

for i, doc in enumerate(results, start=1):
    print(
        f"[{i}] "
        f"page={doc.metadata.get('page')} | "
        f"pages={doc.metadata.get('pages')} | "
        f"heading={doc.metadata.get('heading')}"
    )

    print(doc.page_content[:350].replace("\n", " "))
    print()


[1] page=21 | pages=21 | heading=5.2 AI 기술 토픽으로 본 노력 및 대응 > < 추론형 데이터 종류(예시) >
5.2 AI 기술 토픽으로 본 노력 및 대응 < 추론형 데이터 종류(예시) > ① 단계적 '과정' 추론, 설명 = Ÿ AI가 최종 결론에 도달하는 '과정' 자체를 학습하는 데 중점을 둔 데이터 - 절차적 추론 데이터(CoT), 상호작용적 추론 데이터 등 - (예시) 민원처리 절차의 체크리스트 및 결정 기준. ② '근거' 기반 설명, 설명 = Ÿ AI가 답을 말할 때 어떤 근거를 인용해야 하는지 학습시키기 위한 데이터 - 자연어 설명(NLE) 데이터, 추출적 근거 데이터 등 - (예시) 정책 Q&A 공공데이터 개방률 ? → 포털 통계 페이지 2-3 근거를 인용. ③ '맥락·관계' 이해, 설명 = Ÿ 객체 간의 '관계'나 

[2] page=9 | pages=9 | heading=분석 절차 CHAPTER2 > 3 2.1 자료 수집 절차 및 분석 과정
분석 절차 CHAPTER2 3 2.1 자료 수집 절차 및 분석 과정 - 각 분야별 담론 구조 도출을 위해 본 연구는 데이터 기반 접근법을 적용하였으며, 그 분석 과정은 다음의 단계별 프로세스에 따라 수행되었음

[3] page=8 | pages=8 | heading=1.3 분석 한계 및 유의사항
1.3 분석 한계 및 유의사항 - 본 분석은 2025년 1월부터 11월까지 국내·해외 주요 언론 및 공공·산업 데이터를 중심으로 수행되었으며, 기사·보고서·정책 자료의 주기적 스크리닝과 키워드 정제에 기반하고 있음 - Ÿ 해외 비언론 데이터, 최신 산업·기술 사례, 기업 내부 자료 등은 자료 접근성과 범위 한계로 인해 깊이 있게 반영되지 못하였음 - Ÿ 특정 기간(11개월)의 담론 흐름에 기반하므로, 빠르게 변화하는 AI 환경의 시간적 민감성을 전부 포착하기에는 어려움이 존재함 - 산업·기술·정책 각 분야에서 LDA 토픽모델링을 적용할 때, 데이터 규모와 키워드 밀도, - 담론 구조 등을

# 8. 01번 실습과 비교

| 구분 | 01 Naive RAG | 02 Document Processing |
|---|---|---|
| Loader | `DoclingLoader` | `DoclingLoader` |
| ExportType | `MARKDOWN` | `DOC_CHUNKS` |
| PDF Parsing | Docling | Docling |
| Chunking | `RecursiveCharacterTextSplitter` | `HybridChunker` |
| Chunk 기준 | 문자 길이 중심 | 문서 구조 + token |
| Heading 활용 | 제한적 | Chunk context / metadata 활용 |
| Table 처리 | Chunk 경계에서 분리 가능 | 구조 인식 + header 반복 가능 |
| Page provenance | 제한적 | `dl_meta`에서 추출 가능 |
| 추가 TextSplitter | 필요 | 불필요 |
| Retrieval | Dense | Dense |
| Query Rewrite | 없음 | 없음 |
| Reranking | 없음 | 없음 |

## 핵심 비교

### 01

```text
DoclingLoader
 ↓
MARKDOWN
 ↓
내가 Chunking
```

### 02

```text
DoclingLoader
 ↓
DOC_CHUNKS
 ↓
Docling이 Chunking
```

두 실습 모두 Docling으로 PDF를 파싱하지만,  
**Chunk를 누가 어떻게 만드는가**가 핵심 차이이다.


# 정리

이번 실습의 핵심은 다음과 같다.

1. `DoclingLoader`를 사용해 LangChain과 Docling의 연결을 단순화했다.
2. `ExportType.DOC_CHUNKS`로 Docling의 native Chunking을 사용했다.
3. `HybridChunker`를 이용해 문서 구조와 token 크기를 함께 고려했다.
4. 별도의 `RecursiveCharacterTextSplitter`를 사용하지 않았다.
5. `DoclingLoader`가 생성한 `page_content`를 그대로 Embedding 입력으로 사용했다.
6. `dl_meta`에서 page, heading, caption provenance를 추출했다.
7. Vector DB에서 활용하기 쉽도록 metadata를 평탄화했다.
8. 구조 인식 Chunk를 ChromaDB에 저장하고 검색 결과의 출처를 추적했다.

## 전체 학습 흐름

```text
01 Naive RAG
MARKDOWN + Recursive Chunking
          ↓
02 Document Processing
DOC_CHUNKS + HybridChunker
          ↓
03 Query Rewriting
          ↓
04 Multi Query
          ↓
05 Retrieval Engineering
          ↓
06 Reranking
          ↓
07 Context Engineering
          ↓
08 Grounded Answer
          ↓
09 RAG Evaluation
```

2번 실습의 목적은 **검색 알고리즘을 바꾸기 전에 문서를 더 좋은 검색 단위로 준비하는 것**이다.
